# DS 5001 Final Project

Erin Siedlecki

## Derived Tables

In [3]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from glob import glob
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
nltk.download('stopwords')
np.random.seed(42)

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/erinsiedlecki/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [4]:
LIB = pd.read_csv('../data/LIB.csv', sep='|')
VOCAB = pd.read_csv('../data/VOCAB_base.csv', sep='|', index_col='term_str')
CORPUS = pd.read_csv('../data/CORPUS.csv', sep='|')
OHCO = ['artist', 'song', 'token_num']
CORPUS = CORPUS.set_index(OHCO)

## Bag of Words (BOW)

In [5]:
bag = ['artist', 'song']
BOW = CORPUS.groupby(bag+['term_str']).term_str.count().to_frame('n') 
BOW.index.names = bag+['term_str']
BOW

n
artist        song                           term_str    
Ariana Grande -thank u, next (video version) a          7
                                             about      1
                                             ain        4
                                             aisle      1
                                             all        1
...                                                    ..
Taylor Swift  “Songs Taylor Loves” Playlist  you       13
                                             young      2
                                             your       2
                                             youth      1
                                             zacari     1

[426346 rows x 1 columns]

Bag (expressed in terms of OHCO levels): artist, song

Number of observations: 426,348

Columns (as delimitted names, including n , tfidf ): artist, song, term_str, n, tfidf

## Document-Term Matrix

In [6]:
DTM = BOW.n.unstack(fill_value=0)
DTM

term_str                                                      0  00  000  \
artist        song                                                         
Ariana Grande -thank u, next (video version)                  0   0    0   
              34+35                                           0   0    0   
              34+35 (DJ Siembab Remix)                        0   0    0   
              34+35 (Remix)                                   0   0    0   
              34+35 (Remix) (Clean)                           0   0    0   
...                                                          ..  ..  ...   
Taylor Swift  ​willow (original songwriting demo)             0   0    0   
              ​willow [dancing witch version (Elvira remix)]  0   0    0   
              ​’tis the damn season                           0   0    0   
              “Happy Voting! 🗳😃🌈”                             0   1    0   
              “Songs Taylor Loves” Playlist                   0   0    0   

term_str                                                      00000  000s  \
artist        song                                                          
Ariana Grande -thank u, next (video version)                      0     0   
              34+35                                               0     0   
              34+35 (DJ Siembab Remix)                            0     0   
              34+35 (Remix)                                       0     0   
              34+35 (Remix) (Clean)                               0     0   
...                                                             ...   ...   
Taylor Swift  ​willow (original songwriting demo)                 0     0   
              ​willow [dancing witch version (Elvira remix)]      0     0   
              ​’tis the damn season                               0     0   
              “Happy Voting! 🗳😃🌈”                                 0     0   
              “Songs Taylor Loves” Playlist                       0     0   

term_str                                                      004  005  006  \
artist        song                                                            
Ariana Grande -thank u, next (video version)                    0    0    0   
              34+35                                             0    0    0   
              34+35 (DJ Siembab Remix)                          0    0    0   
              34+35 (Remix)                                     0    0    0   
              34+35 (Remix) (Clean)                             0    0    0   
...                                                           ...  ...  ...   
Taylor Swift  ​willow (original songwriting demo)               0    0    0   
              ​willow [dancing witch version (Elvira remix)]    0    0    0   
              ​’tis the damn season                             0    0    0   
              “Happy Voting! 🗳😃🌈”                               0    0    0   
              “Songs Taylor Loves” Playlist                     0    0    0   

term_str                                                      007  008  ...  \
artist        song                                                      ...   
Ariana Grande -thank u, next (video version)                    0    0  ...   
              34+35                                             0    0  ...   
              34+35 (DJ Siembab Remix)                          0    0  ...   
              34+35 (Remix)                                     0    0  ...   
              34+35 (Remix) (Clean)                             0    0  ...   
...                                                           ...  ...  ...   
Taylor Swift  ​willow (original songwriting demo)               0    0  ...   
              ​willow [dancing witch version (Elvira remix)]    0    0  ...   
              ​’tis the damn season                             0    0  ...   
              “Happy Voting! 🗳😃🌈”                               0    0  ...   
              “Songs Taylor Loves

In [7]:
DTM.to_csv('../data/DTM.csv', sep='|', index=True)

Bag: artist, song

## TFIDF

In [8]:
TF = DTM.T / DTM.T.sum()
TF = TF.T
TF.head(5)

term_str                                        0   00  000  00000  000s  004  \
artist        song                                                              
Ariana Grande -thank u, next (video version)  0.0  0.0  0.0    0.0   0.0  0.0   
              34+35                           0.0  0.0  0.0    0.0   0.0  0.0   
              34+35 (DJ Siembab Remix)        0.0  0.0  0.0    0.0   0.0  0.0   
              34+35 (Remix)                   0.0  0.0  0.0    0.0   0.0  0.0   
              34+35 (Remix) (Clean)           0.0  0.0  0.0    0.0   0.0  0.0   

term_str                                      005  006  007  008  ...  zro  \
artist        song                                                ...        
Ariana Grande -thank u, next (video version)  0.0  0.0  0.0  0.0  ...  0.0   
              34+35                           0.0  0.0  0.0  0.0  ...  0.0   
              34+35 (DJ Siembab Remix)        0.0  0.0  0.0  0.0  ...  0.0   
              34+35 (Remix)                   0.0  0.0  0.0  0.0  ...  0.0   
              34+35 (Remix) (Clean)           0.0  0.0  0.0  0.0  ...  0.0   

term_str                                       zu  zub  zuboomboom  zucchinis  \
artist        song                                                              
Ariana Grande -thank u, next (video version)  0.0  0.0         0.0        0.0   
              34+35                           0.0  0.0         0.0        0.0   
              34+35 (DJ Siembab Remix)        0.0  0.0         0.0        0.0   
              34+35 (Remix)                   0.0  0.0         0.0        0.0   
              34+35 (Remix) (Clean)           0.0  0.0         0.0        0.0   

term_str                                      zurich  zuse  zuzu  zwrotka   zz  
artist        song                                                              
Ariana Grande -thank u, next (video version)     0.0   0.0   0.0      0.0  0.0  
              34+35                              0.0   0.0   0.0      0.0  0.0  
              34+35 (DJ Siembab Remix)           0.0   0.0   0.0      0.0  0.0  
              34+35 (Remix)                      0.0   0.0   0.0      0.0  0.0  
              34+35 (Remix) (Clean)              0.0   0.0   0.0      0.0  0.0  

[5 rows x 20191 columns]

In [9]:
DF = DTM.astype('bool').sum() 
DF.head(5)

term_str
0        105
00        32
000        6
00000      1
000s       1
dtype: int64

In [10]:
N = DTM.shape[0]
IDF = np.log2(N / DF)
IDF.head(5)

term_str
0         5.288921
00        7.003167
000       9.418204
00000    12.003167
000s     12.003167
dtype: float64

In [11]:
TFIDF = TF * IDF
TFIDF

term_str                                                        0        00  \
artist        song                                                            
Ariana Grande -thank u, next (video version)                  0.0  0.000000   
              34+35                                           0.0  0.000000   
              34+35 (DJ Siembab Remix)                        0.0  0.000000   
              34+35 (Remix)                                   0.0  0.000000   
              34+35 (Remix) (Clean)                           0.0  0.000000   
...                                                           ...       ...   
Taylor Swift  ​willow (original songwriting demo)             0.0  0.000000   
              ​willow [dancing witch version (Elvira remix)]  0.0  0.000000   
              ​’tis the damn season                           0.0  0.000000   
              “Happy Voting! 🗳😃🌈”                             0.0  0.017865   
              “Songs Taylor Loves” Playlist                   0.0  0.000000   

term_str                                                      000  00000  \
artist        song                                                         
Ariana Grande -thank u, next (video version)                  0.0    0.0   
              34+35                                           0.0    0.0   
              34+35 (DJ Siembab Remix)                        0.0    0.0   
              34+35 (Remix)                                   0.0    0.0   
              34+35 (Remix) (Clean)                           0.0    0.0   
...                                                           ...    ...   
Taylor Swift  ​willow (original songwriting demo)             0.0    0.0   
              ​willow [dancing witch version (Elvira remix)]  0.0    0.0   
              ​’tis the damn season                           0.0    0.0   
              “Happy Voting! 🗳😃🌈”                             0.0    0.0   
              “Songs Taylor Loves” Playlist                   0.0    0.0   

term_str                                                      000s  004  005  \
artist        song                                                             
Ariana Grande -thank u, next (video version)                   0.0  0.0  0.0   
              34+35                                            0.0  0.0  0.0   
              34+35 (DJ Siembab Remix)                         0.0  0.0  0.0   
              34+35 (Remix)                                    0.0  0.0  0.0   
              34+35 (Remix) (Clean)                            0.0  0.0  0.0   
...                                                            ...  ...  ...   
Taylor Swift  ​willow (original songwriting demo)              0.0  0.0  0.0   
              ​willow [dancing witch version (Elvira remix)]   0.0  0.0  0.0   
              ​’tis the damn season                            0.0  0.0  0.0   
              “Happy Voting! 🗳😃🌈”                              0.0  0.0  0.0   
              “Songs Taylor Loves” Playlist                    0.0  0.0  0.0   

term_str                                                      006  007  008  \
artist        song                                                            
Ariana Grande -thank u, next (video version)                  0.0  0.0  0.0   
              34+35                                           0.0  0.0  0.0   
              34+35 (DJ Siembab Remix)                        0.0  0.0  0.0   
              34+35 (Remix)                                   0.0  0.0  0.0   
              34+35 (Remix) (Clean)                           0.0  0.0  0.0   
...                                                           ...  ...  ...   
Taylor Swift  ​willow (original songwriting demo)             0.0  0.0  0.0   
              ​willow [dancing witch version (Elvira remix)]  0.0  0.0  0.0   
              ​’tis the damn season                           0.0  0.0  0.0   
              “Happy Voting! 🗳😃🌈”                             0.0  0.0  0

In [12]:
TFIDF.to_csv('../data/TFIDF.csv', sep='|', index=True)

In [13]:
VOCAB['df'] = DF
VOCAB['idf'] = IDF
VOCAB['dfidf'] = VOCAB['df'] * VOCAB['idf']
VOCAB['tfidf_mean'] = TFIDF.mean()
BOW['tfidf'] = TFIDF.stack()

Description of TFIDIF formula: To calculate the TFIDF values, I multiplied the TF value by the IDF value. The TF values were computed using the sum method, and the IDF values were computed using the standard method.

In [14]:
top20 = (VOCAB[VOCAB['stop'] == False].sort_values('dfidf', ascending=False)
         .head(20).index.tolist())
top20

['let',
 'time',
 'one',
 'see',
 'say',
 'never',
 'baby',
 'get',
 'yeah',
 'pre',
 'want',
 'go',
 'got',
 'make',
 'way',
 'wanna',
 'take',
 'back',
 'come',
 'need']

Top 20 significant words in the corpus by DFIDF: ['let', 'time', 'one', 'see', 'say','never', 'baby', 'get', 'yeah', 'pre', 'want', 'go', 'got', 'make', 'way', 'wanna', 'take', 'back', 'come', 'need']

In [15]:
VOCAB.to_csv('../data/VOCAB.csv', sep='|', index=True)
BOW.to_csv('../data/BOW.csv', sep='|', index=True)

## Reduced and Normalized TFIDF_L2

In [16]:
from sklearn.preprocessing import normalize

sigs = VOCAB[(VOCAB['stop'] == False) & (VOCAB['df']>=5)].index
TFIDF_red = TFIDF[sigs]

TFIDF_L2 = pd.DataFrame(
    normalize(TFIDF_red, norm='l2'),
    index=TFIDF_red.index,
    columns=TFIDF_red.columns
)
TFIDF_L2

term_str                                                        0       00  \
artist        song                                                           
Ariana Grande -thank u, next (video version)                  0.0  0.00000   
              34+35                                           0.0  0.00000   
              34+35 (DJ Siembab Remix)                        0.0  0.00000   
              34+35 (Remix)                                   0.0  0.00000   
              34+35 (Remix) (Clean)                           0.0  0.00000   
...                                                           ...      ...   
Taylor Swift  ​willow (original songwriting demo)             0.0  0.00000   
              ​willow [dancing witch version (Elvira remix)]  0.0  0.00000   
              ​’tis the damn season                           0.0  0.00000   
              “Happy Voting! 🗳😃🌈”                             0.0  0.06394   
              “Songs Taylor Loves” Playlist                   0.0  0.00000   

term_str                                                      000   04   05  \
artist        song                                                            
Ariana Grande -thank u, next (video version)                  0.0  0.0  0.0   
              34+35                                           0.0  0.0  0.0   
              34+35 (DJ Siembab Remix)                        0.0  0.0  0.0   
              34+35 (Remix)                                   0.0  0.0  0.0   
              34+35 (Remix) (Clean)                           0.0  0.0  0.0   
...                                                           ...  ...  ...   
Taylor Swift  ​willow (original songwriting demo)             0.0  0.0  0.0   
              ​willow [dancing witch version (Elvira remix)]  0.0  0.0  0.0   
              ​’tis the damn season                           0.0  0.0  0.0   
              “Happy Voting! 🗳😃🌈”                             0.0  0.0  0.0   
              “Songs Taylor Loves” Playlist                   0.0  0.0  0.0   

term_str                                                            08   09  \
artist        song                                                            
Ariana Grande -thank u, next (video version)                  0.000000  0.0   
              34+35                                           0.000000  0.0   
              34+35 (DJ Siembab Remix)                        0.000000  0.0   
              34+35 (Remix)                                   0.000000  0.0   
              34+35 (Remix) (Clean)                           0.000000  0.0   
...                                                                ...  ...   
Taylor Swift  ​willow (original songwriting demo)             0.000000  0.0   
              ​willow [dancing witch version (Elvira remix)]  0.000000  0.0   
              ​’tis the damn season                           0.000000  0.0   
              “Happy Voting! 🗳😃🌈”                             0.000000  0.0   
              “Songs Taylor Loves” Playlist                   0.062235  0.0   

term_str                                                        4   40  \
artist        song                                                       
Ariana Grande -thank u, next (video version)                  0.0  0.0   
              34+35                                           0.0  0.0   
              34+35 (DJ Siembab Remix)                        0.0  0.0   
              34+35 (Remix)                                   0.0  0.0   
              34+35 (Remix) (Clean)                           0.0  0.0   
...                                                           ...  ...   
Taylor Swift  ​willow (original songwriting demo)             0.0  0.0   
              ​willow [dancing witch version (Elvira remix)]  0.0  0.0   
              ​’tis the damn season                           0.0  0.0   
              “Happy Voting! 🗳😃🌈”                             0.0  0.0   
              “Songs Taylor Loves” Playli

Number of features (i.e. significant words): 5,669

Principle of significant word selection: I selected the significant words by removing stopwords and filtering for words with df >= 5 in order to keep only the higher-frequency, content-bearing words.

In [17]:
TFIDF_L2.to_csv('../data/TFIDF_L2.csv', sep='|', index=True)